# Mock Supply Chain Data Generator

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

### Set seed for reproducibility

In [4]:
np.random.seed(42)

#### 1. DIMENSION: Calendar / Dates (Jan 2025 to Dec 2026)

In [5]:
start_date = datetime(2025, 1, 1)
end_date = datetime(2026, 12, 31)
date_range = pd.date_range(start_date, end_date)

dim_calendar = pd.DataFrame({
    'Date_Key': date_range.strftime('%Y%m%d').astype(int),
    'Date': date_range,
    'Year': date_range.year,
    'Quarter': date_range.quarter,
    'Month_Name': date_range.strftime('%B'),
    'Month_Num': date_range.month,
    'Is_Weekend': date_range.weekday.isin([5, 6]).astype(int)
})

#### 2. DIMENSION: Products

In [6]:
products = [
    ('P001', 'Eco Lithium Battery Pack', 'Components', 120.00, 150),
    ('P002', 'Copper Wiring Harness 2m', 'Raw Materials', 15.50, 500),
    ('P003', 'Aluminium Housing Shell', 'Structural', 45.00, 200),
    ('P004', 'Smart Microcontroller V2', 'Electronics', 85.25, 100),
    ('P005', 'High-Tensile Steel Bolts', 'Fasteners', 2.10, 1200)
]

dim_products = pd.DataFrame(products, columns=[
    'Product_Key', 'Product_Name', 'Category', 'Unit_Cost', 'Reorder_Point'
])

#### 3. DIMENSION: Suppliers

In [7]:
suppliers = [
    ('S01', 'Alpha Logistics & Parts', 'South', 5),  # 5 days agreed lead time
    ('S02', 'Global Tech Components', 'West', 7),
    ('S03', 'Apex Foundry & Steel', 'North', 12),
    ('S04', 'Zenith Micro Electronics', 'East', 15)
]

dim_suppliers = pd.DataFrame(suppliers, columns=[
    'Supplier_Key', 'Supplier_Name', 'Region', 'Agreed_Lead_Time'
])

#### 4. FACT: Inventory Movements & Logistics

In [8]:
# Generate 5,000 random transaction records
num_records = 5000

In [9]:
# Randomly sample dates, products, and suppliers
fact_dates = np.random.choice(dim_calendar['Date'], size=num_records)
fact_products = np.random.choice(dim_products['Product_Key'], size=num_records)
fact_suppliers = np.random.choice(dim_suppliers['Supplier_Key'], size=num_records)

fact_table = pd.DataFrame({
    'Transaction_ID': [f'TXN{i:06d}' for i in range(1, num_records + 1)],
    'Date': fact_dates,
    'Product_Key': fact_products,
    'Supplier_Key': fact_suppliers
})

In [10]:
# Add Date_Key for the Star Schema relationship
fact_table['Date_Key'] = fact_table['Date'].dt.strftime('%Y%m%d').astype(int)

In [11]:
# Quantities Sold & Ordered (Mimicking demand and supply)
fact_table['Quantity_Sold'] = np.random.randint(5, 80, size=num_records)
fact_table['Quantity_Ordered'] = np.random.randint(50, 200, size=num_records)

In [12]:
# Create an intentional pattern: Make Supplier S04 habitually late to give you data to analyze!
def generate_lead_time(supplier):
    if supplier == 'S04':
        return int(np.random.normal(loc=19, scale=4)) # Agreed is 15, but mean is 19 (High Risk!)
    else:
        return max(2, int(np.random.normal(loc=6, scale=2)))

fact_table['Actual_Lead_Time_Days'] = fact_table['Supplier_Key'].apply(generate_lead_time)

In [13]:
# Drop raw date from fact table to keep data modeling clean
fact_table.drop(columns=['Date'], inplace=True)

#### Export to CSV files for Power BI Import

In [14]:
dim_calendar.to_csv('Dim_Calendar.csv', index=False)
dim_products.to_csv('Dim_Products.csv', index=False)
dim_suppliers.to_csv('Dim_Suppliers.csv', index=False)
fact_table.to_csv('Fact_Inventory_Movements.csv', index=False)

print("✅ Portfolio project dataset generated successfully! 4 CSV files saved.")

✅ Portfolio project dataset generated successfully! 4 CSV files saved.
